# 11 — Explicabilidade (TEA): o que o modelo usa, e se isso é estável

> **Este notebook é exploratório.** Atribuições mostram o que o modelo usa, **não** causalidade nem mecanismo biológico. Só faz sentido interpretar quando o modelo está acima do acaso; por isso há um **portão** (célula 4) que decide isso com o mesmo critério dos notebooks 08–10 (IC 95% da AUC OOF, por bootstrap de sites, com limite inferior > 0,5).

**O que é feito (tudo por fold externo, com o modelo ajustado só no treino e explicado no fold de teste)**

| Etapa | Método | Por que |
|---|---|---|
| Modelo a explicar | O baseline de maior AUC média no notebook 08 (`MODELO_XAI = 'auto'`), reajustado por fold com os hiperparâmetros registrados em `08_resultados_long.csv` | Não existe "modelo final em todos os dados" aqui; explicar um modelo treinado no conjunto inteiro misturaria treino e teste |
| Portão | IC 95% da AUC OOF (bootstrap de sites) do modelo e de um modelo linear de referência | Sem desempenho acima do acaso, explicar é interpretar ruído |
| Estabilidade (modelo linear de referência) | Regressão logística L2 em atributos padronizados só com o treino; as 100 conexões de maior valor absoluto do coeficiente em cada fold; quantos folds repetem cada conexão; consistência de sinal | Mostra se algo se repete entre folds |
| Referência de acaso | Mesma análise com **rótulos de treino embaralhados** | Se o real não for mais estável que o embaralhado, não há o que interpretar |
| Importância por ROI (modelo escolhido) | Permutação em grupo: embaralha, entre os participantes do fold de teste, todas as conexões de uma ROI e mede a queda da AUC | Funciona com qualquer modelo (SVM, RF, XGBoost) e fica no nível de ROI, onde há mais sinal por unidade |

**Limites que devem aparecer no texto do TCC**
- Os folds compartilham ~75% do treino; a estabilidade entre folds é, por isso, otimista.
- O ordenamento das conexões segue o notebook 07 (triângulo superior, por linha). As ROIs são identificadas pela posição na matriz (ou pelo id do atlas, quando disponível); nomear regiões anatômicas exige cruzar com o atlas CC200.
- Efeitos de site, movimento e protocolo podem aparecer como "importância". Este notebook **não** remove esses efeitos.
- Deep learning e grafos não são explicados aqui: em geral não superam os baselines nesta base, e atribuições de modelos no acaso não têm valor. Se algum deles passar do acaso nos resultados, a explicação deles (Integrated Gradients / atenção do GAT) é uma extensão a decidir depois.

In [ ]:
# 1. Projeto, módulo de avaliação e dataset (TEA / ABIDE I)
import sys, warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

_raiz = Path.cwd().resolve()
for _c in (_raiz, *_raiz.parents):
    if (_c / 'src' / 'config.py').exists():
        _raiz = _c
        break
else:
    raise RuntimeError('Não encontrei src/config.py. Abra a pasta TCC_TEA_TDAH no VS Code.')
if str(_raiz) not in sys.path:
    sys.path.insert(0, str(_raiz))

from src.config import BASE_DIR, PATHS, SEED, criar_pastas
from src import avaliacao as av
criar_pastas()

NOME_BASE = 'TEA'
ROTULO_CASO = 'TEA'
ARQ_DATASET = PATHS['processed'] / 'dataset_abide_processado.pkl'
PASTA_RES = Path(PATHS['results'])
PASTA_MOD = Path(PATHS['models'])
PASTA_RES.mkdir(parents=True, exist_ok=True)
PASTA_MOD.mkdir(parents=True, exist_ok=True)

PASTA_FIG = Path(PATHS['figures']); PASTA_FIG.mkdir(parents=True, exist_ok=True)
# 2. Parâmetros desta execução
import ast, json
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

MODELO_XAI = 'auto'          # 'auto' = baseline de maior AUC média entre folds (notebook 08) | 'SVM_RBF' | 'Random_Forest' | 'XGBoost'
N_REPETICOES = 3             # permutações por ROI (importância por permutação em grupo)
TOP_K = 100                  # conexões "selecionadas" por fold no modelo linear
C_LINEAR = 0.01              # regularização L2 fixa do modelo linear de referência (não é ajustada no teste)
N_NULOS = 3                  # réplicas com rótulos de treino embaralhados (referência de acaso)
FORCAR_XAI = False           # True: roda a permutação por ROI mesmo se o modelo não passar do acaso (exploratório)
if av.RAPIDO:
    N_REPETICOES, N_NULOS = 2, 1
    print('*** TCC_RAPIDO=1: teste curto ***')

d = av.carregar_dataset(ARQ_DATASET)
X, y, sites, fold_id = d['X'], d['y'], d['sites'], d['fold_id']
N = av.n_rois_de(X.shape[1])
ri, rj = np.triu_indices(N, k=1)          # mesma ordem de atributos do notebook 07 (triângulo superior, por linha)
folds = sorted(np.unique(fold_id).tolist())
pd.set_option('display.width', 200)
ROI_ROTULOS = [f'ROI {k + 1}' for k in range(N)]
print('ROIs rotuladas pela posição na matriz (1..N), na ordem do notebook 07; para nomear regiões, cruze com o atlas CC200.')
print(f'Base: {NOME_BASE} | participantes: {len(y)} | ROIs: {N} | conexões: {X.shape[1]} | folds: {folds}')

In [ ]:
# 3. Modelo a explicar e hiperparâmetros por fold (vêm do notebook 08)
arq08 = PASTA_RES / '08_resultados_long.csv'
if not arq08.exists():
    raise FileNotFoundError('08_resultados_long.csv não encontrado: rode o notebook 08 antes deste.')
r08 = pd.read_csv(arq08)
r08 = r08[~r08['modelo'].str.startswith('Dummy')]
if MODELO_XAI == 'auto':
    MODELO_XAI = r08.groupby('modelo')['auc_roc'].mean().idxmax()
sem_ref = int(r08['semente'].min())        # semente cujas previsões OOF o 08 guardou
PARAMS, auc_08 = {}, {}
for f in folds:
    linha = r08[(r08['modelo'] == MODELO_XAI) & (r08['fold'] == f) & (r08['semente'] == sem_ref)]
    auc_08[f] = float(linha['auc_roc'].iloc[0])
    try:
        PARAMS[f] = ast.literal_eval(linha['params'].iloc[0])
    except Exception:
        PARAMS[f] = {}
        print(f'AVISO: não consegui ler os hiperparâmetros do fold {f}; usando os padrão.')
print('Modelo explicado:', MODELO_XAI, '| semente de referência:', sem_ref)
display(pd.DataFrame({'hiperparâmetros (08)': {f: str(PARAMS[f]) for f in folds}}))

In [ ]:
# 4. Reajuste por fold (só treino), desempenho fora do treino e PORTÃO de interpretação
modelos_fold, coef_lin = {}, {}
proba_m = np.full(len(y), np.nan)
proba_lin = np.full(len(y), np.nan)
linhas = []
for f in folds:
    te = np.where(fold_id == f)[0]; tr = np.where(fold_id != f)[0]
    t0 = time.time()
    m = av.criar_modelo(MODELO_XAI, y[tr], seed=sem_ref, params=PARAMS[f]).fit(X[tr], y[tr])
    modelos_fold[f] = m
    proba_m[te] = av.pontuar(m, X[te])
    sc = StandardScaler().fit(X[tr])
    lin = LogisticRegression(C=C_LINEAR, class_weight='balanced', max_iter=1000).fit(sc.transform(X[tr]), y[tr])
    proba_lin[te] = lin.predict_proba(sc.transform(X[te]))[:, 1]
    coef_lin[f] = lin.coef_[0].copy()
    linhas.append(dict(fold=f, n_teste=len(te), auc_modelo=av.roc_auc_score(y[te], proba_m[te]),
                       auc_08=auc_08[f], auc_linear=av.roc_auc_score(y[te], proba_lin[te]),
                       seg=round(time.time() - t0, 1)))
tab_aj = pd.DataFrame(linhas)
tab_aj['dif_vs_08'] = tab_aj['auc_modelo'] - tab_aj['auc_08']
print('AUC por fold (modelo reajustado aqui x valor registrado no 08; o modelo linear é só referência)')
display(tab_aj.round(3))
print('Diferenças pequenas vs. o 08 são esperadas (mesmo modelo e hiperparâmetros, outro ajuste). Diferença grande = investigar.')

ic_m = av.bootstrap_auc_por_site(y, proba_m, sites)
ic_l = av.bootstrap_auc_por_site(y, proba_lin, sites)
passa = bool(ic_m['ic_inf'] > 0.5)
print(f"\n{MODELO_XAI}: AUC OOF = {ic_m['auc']:.3f}  IC95% (sites) = [{ic_m['ic_inf']:.3f}, {ic_m['ic_sup']:.3f}]")
print(f"Modelo linear de referência: AUC OOF = {ic_l['auc']:.3f}  IC95% = [{ic_l['ic_inf']:.3f}, {ic_l['ic_sup']:.3f}]")
print('=' * 78)
if passa:
    print('PORTÃO: IC inferior > 0,5 -> as explicações abaixo podem ser lidas como HIPÓTESES exploratórias.')
else:
    print('PORTÃO: o IC inclui 0,5 -> o modelo NÃO é distinguível do acaso. As explicações abaixo descrevem')
    print('        o que o modelo usou para ajustar o treino, NÃO um padrão neurobiológico. Não interpretar.')
print('=' * 78)

In [ ]:
# 5. Modelo linear de referência: estabilidade entre folds, real x rótulos embaralhados
def resumo_estabilidade(coefs):
    """coefs: lista com um vetor de coeficientes por fold."""
    F = len(coefs)
    S = np.vstack(coefs)
    A = np.abs(S)
    sel = np.zeros(A.shape[1], dtype=int)
    for a in A:
        sel[np.argsort(a)[::-1][:TOP_K]] += 1
    limiar = int(np.ceil(0.8 * F))
    roi = np.vstack([np.bincount(ri, weights=a, minlength=N) + np.bincount(rj, weights=a, minlength=N) for a in A])
    rhos = [spearmanr(roi[a], roi[b])[0] for a in range(F) for b in range(a + 1, F)]
    est = sel >= limiar
    mesmo_sinal = np.abs(np.sign(S).sum(0)) == F
    return dict(sel=sel, roi=roi, S=S, limiar=limiar,
                n_conexoes_estaveis=int(est.sum()), n_estaveis_mesmo_sinal=int((est & mesmo_sinal).sum()),
                max_folds_numa_conexao=int(sel.max()), rho_roi_medio=float(np.nanmean(rhos)))

real = resumo_estabilidade([coef_lin[f] for f in folds])
rng_n = np.random.default_rng(SEED)
linhas_est = [dict(analise='rótulos reais', **{k: real[k] for k in ['n_conexoes_estaveis', 'n_estaveis_mesmo_sinal', 'max_folds_numa_conexao', 'rho_roi_medio']})]
for k in range(N_NULOS):
    coefs_n = []
    for f in folds:
        tr = np.where(fold_id != f)[0]
        sc = StandardScaler().fit(X[tr])
        yp = rng_n.permutation(y[tr])
        ln = LogisticRegression(C=C_LINEAR, class_weight='balanced', max_iter=1000).fit(sc.transform(X[tr]), yp)
        coefs_n.append(ln.coef_[0].copy())
    nul = resumo_estabilidade(coefs_n)
    linhas_est.append(dict(analise=f'rótulos embaralhados #{k + 1}', **{c: nul[c] for c in ['n_conexoes_estaveis', 'n_estaveis_mesmo_sinal', 'max_folds_numa_conexao', 'rho_roi_medio']}))
tab_est = pd.DataFrame(linhas_est)
print(f'Estável = conexão entre as {TOP_K} maiores |coef| em pelo menos {real["limiar"]} de {len(folds)} folds.')
display(tab_est.round(3))
print('Leitura: se "rótulos reais" não superar claramente os embaralhados, não há padrão estável a interpretar.')
print('Atenção: os treinos dos folds se sobrepõem (~75%), então a estabilidade real é otimista.')

In [ ]:
# 6. Importância por permutação em grupo (por ROI), no fold de teste, para o modelo escolhido
imp_perm = None
if passa or FORCAR_XAI:
    cols_roi = [np.where((ri == r) | (rj == r))[0] for r in range(N)]
    rng_p = np.random.default_rng(SEED)
    imp_perm = np.zeros((len(folds), N))
    for a, f in enumerate(folds):
        te = np.where(fold_id == f)[0]
        Xte, yte, m = X[te].copy(), y[te], modelos_fold[f]
        base = av.roc_auc_score(yte, av.pontuar(m, Xte))
        t0 = time.time()
        for r in range(N):
            c = cols_roi[r]
            guarda = Xte[:, c].copy()
            quedas = []
            for _ in range(N_REPETICOES):
                Xte[:, c] = guarda[rng_p.permutation(len(te))]
                quedas.append(base - av.roc_auc_score(yte, av.pontuar(m, Xte)))
            Xte[:, c] = guarda
            imp_perm[a, r] = float(np.mean(quedas))
        print(f'fold {f}: AUC de teste {base:.3f} | {N} ROIs x {N_REPETICOES} permutações em {(time.time() - t0) / 60:.1f} min')
    print('Importância = queda média da AUC ao embaralhar a ROI (positiva = o modelo dependia dela). Quedas ~0 ou negativas = sem uso.')
else:
    print('Pulado: o modelo não passou do acaso no portão (célula 4). Para rodar mesmo assim, FORCAR_XAI = True (exploratório).')

In [ ]:
# 7. Tabelas e figuras
S = real['S']
abs_med = np.abs(S).mean(0)
tab_con = pd.DataFrame(dict(ROI_a=[ROI_ROTULOS[k] for k in ri], ROI_b=[ROI_ROTULOS[k] for k in rj],
                            folds_no_topK=real['sel'], coef_medio=S.mean(0), abs_coef_medio=abs_med,
                            mesmo_sinal_em_todos=(np.abs(np.sign(S).sum(0)) == len(folds))))
tab_con = tab_con.sort_values(['folds_no_topK', 'abs_coef_medio'], ascending=False).reset_index(drop=True)
print('CONEXÕES MAIS REPETIDAS ENTRE FOLDS (modelo linear de referência; coef > 0: conexão mais forte => maior probabilidade do caso)')
display(tab_con.head(20).round(4))

imp_lin = real['roi'].mean(0)
tab_roi = pd.DataFrame(dict(roi=ROI_ROTULOS, importancia_linear=imp_lin,
                            posicao_linear=np.argsort(np.argsort(-imp_lin)) + 1))
if imp_perm is not None:
    tab_roi['queda_auc_permutacao'] = imp_perm.mean(0)
    tab_roi['folds_com_queda_positiva'] = (imp_perm > 0).sum(0)
tab_roi = tab_roi.sort_values('queda_auc_permutacao' if imp_perm is not None else 'importancia_linear', ascending=False).reset_index(drop=True)
print('\nROIs (importância por permutação do modelo escolhido, quando rodou; e |coef| somado do modelo linear)')
display(tab_roi.head(15).round(4))

fig, ax = plt.subplots(1, 2 if imp_perm is not None else 1, figsize=(14 if imp_perm is not None else 7, 5.5), squeeze=False)
ax = ax[0]
t = tab_roi.sort_values('importancia_linear', ascending=False).head(15).iloc[::-1]
ax[0].barh(t['roi'], t['importancia_linear']); ax[0].set_title('Modelo linear: |coef| somado por ROI (média dos folds)')
if imp_perm is not None:
    t = tab_roi.head(15).iloc[::-1]
    ix = [ROI_ROTULOS.index(r) for r in t['roi']]
    ax[1].barh(t['roi'], t['queda_auc_permutacao'], xerr=imp_perm[:, ix].std(0), alpha=.8)
    ax[1].axvline(0, color='k', lw=.8); ax[1].set_title(f'{MODELO_XAI}: queda da AUC ao embaralhar a ROI')
if not passa:
    fig.suptitle('EXPLORATÓRIO — modelo não distinguível do acaso: não interpretar', color='firebrick')
plt.tight_layout(); plt.savefig(PASTA_FIG / '11_xai_roi.png', dpi=200); plt.show()

M = np.zeros((N, N)); M[ri, rj] = S.mean(0); M = M + M.T
lim = np.percentile(np.abs(M[np.triu_indices(N, 1)]), 99) or 1.0
plt.figure(figsize=(6.5, 5.5)); plt.imshow(M, cmap='RdBu_r', vmin=-lim, vmax=lim); plt.colorbar(label='coef médio (padronizado)')
plt.title('Modelo linear: coeficiente médio por conexão' + ('' if passa else '\n(exploratório — sem sinal acima do acaso)'))
plt.xlabel('ROI'); plt.ylabel('ROI'); plt.tight_layout(); plt.savefig(PASTA_FIG / '11_xai_conexoes.png', dpi=200); plt.show()

# 8. Salvar
tab_aj.to_csv(PASTA_RES / '11_ajuste_por_fold.csv', index=False)
tab_est.to_csv(PASTA_RES / '11_estabilidade_real_vs_nulo.csv', index=False)
tab_con.head(200).to_csv(PASTA_RES / '11_conexoes_linear_top200.csv', index=False)
tab_roi.to_csv(PASTA_RES / '11_importancia_roi.csv', index=False)
with open(PASTA_RES / '11_portao_xai.json', 'w', encoding='utf-8') as fh:
    json.dump(dict(base=NOME_BASE, modelo=MODELO_XAI, auc_oof=ic_m['auc'], ic_inf=ic_m['ic_inf'], ic_sup=ic_m['ic_sup'],
                   acima_do_acaso=passa, auc_oof_linear=ic_l['auc'], permutacao_rodou=imp_perm is not None), fh, indent=2, ensure_ascii=False)
print('Salvo em:', PASTA_RES, 'e figuras em', PASTA_FIG)

## Como relatar (sugestão)
- Diga qual modelo foi explicado, que ele foi reajustado por fold e que as explicações vêm de folds de teste por site.
- Reporte o portão: AUC OOF e IC. Se o IC incluir 0,5, escreva que o modelo não se distingue do acaso e que a análise é descritiva; **não** nomeie regiões como "marcadores" do transtorno.
- Reporte a estabilidade real **junto** com a dos rótulos embaralhados, e que os treinos dos folds se sobrepõem.
- Mesmo com sinal, atribuições são hipóteses: dependem do modelo, podem refletir site, idade, movimento ou protocolo, e não indicam causa.